# 🧠 Elite Kaggle Brain for GitHub Actions Agent
This notebook serves as the hyper-optimized LLM backend for your GitHub Actions Autonomous Agent.
**INSTRUCTIONS:**
1. Turn on the **T4x2 GPUs** in the Kaggle Accelerator settings.
2. Turn on **Internet** in the Kaggle settings.
3. Paste your Ngrok Authtoken in the last cell.
4. Run all cells.
5. Copy the Ngrok URL printed at the bottom and paste it into your GitHub Repository Secrets as `OLLAMA_NGROK_URL`.

In [ ]:
!sudo apt-get update -y && sudo apt-get install -y zstd
!pip install pyngrok -q
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os
import subprocess
import time

# Elite Environment Variables for T4x2 (32GB VRAM)
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'
os.environ['OLLAMA_NUM_PARALLEL'] = '1'
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'
os.environ['OLLAMA_MAX_VRAM'] = '31000000000'

# Start Ollama server in the background
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3) # Wait for server to boot

MODEL_NAME = 'qwen2.5-coder:32b-instruct-q5_K_M'
print(f"Pulling {MODEL_NAME}... This might take a few minutes.")
!ollama pull $MODEL_NAME
print("Model pulled successfully!")

In [ ]:
# Create strict JSON tool-calling agent modelfile
modelfile_content = '''FROM qwen2.5-coder:32b-instruct-q5_K_M
PARAMETER temperature 0.6
PARAMETER top_p 0.9
SYSTEM "You are an elite autonomous coding agent. When using tools, you must output valid JSON strictly conforming to the tool schema. Do not add markdown formatting around JSON."'''

with open('Modelfile', 'w') as f:
    f.write(modelfile_content)

!ollama create qwen-coder-agent -f Modelfile
print("qwen-coder-agent successfully created!")

In [ ]:
from pyngrok import ngrok
import subprocess

# ⚠️ PASTE YOUR NGROK AUTHTOKEN HERE
NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"

ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Use subprocess to launch ngrok with the specific request-header-add argument to bypass Kaggle proxies
ngrok_process = subprocess.Popen(['ngrok', 'http', '11434', '--request-header-add', 'host: "localhost:11434"'])
time.sleep(5)

import requests
try:
    response = requests.get("http://localhost:4040/api/tunnels")
    tunnels = response.json()["tunnels"]
    public_url = tunnels[0]["public_url"]
    api_url = f"{public_url}/v1"
    print("\n" + "="*60)
    print("🚀 YOUR ELITE KAGGLE BRAIN IS ONLINE!")
    print("="*60)
    print("Copy this exact URL and put it in your GitHub Secrets as OLLAMA_NGROK_URL:")
    print(f"\n{api_url}\n")
    print("="*60)
    print("Keep this Kaggle tab open to keep the Brain alive.")
except Exception as e:
    print("Failed to get Ngrok URL. Ensure ngrok is running.")
